In [ ]:
import requests
import json
from typing import List, Dict
import PyPDF2 # 導入 PyPDF2 函式庫
import os # 導入 os 模組用於檢查檔案路徑

class OllamaChatbot:
    def __init__(self, model_name="llama3:8b", base_url="http://localhost:11434"):
        """
        初始化聊天機器人
        
        Args:
            model_name: Ollama模型名稱 (預設: llama3:8b)
            base_url: Ollama服務的基礎URL (預設: http://localhost:11434)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.conversation_history: List[Dict[str, str]] = []
        self.attached_pdf_content: str = "" # 新增一個屬性來儲存附加的 PDF 內容
        self.pdf_attached_flag: bool = False # 標記是否有 PDF 內容被附加

    def _extract_text_from_txt(self, txt_path: str) -> str:
        """
        從 TXT 檔案中讀取文字內容。
        """
        try:
            with open(txt_path, 'r', encoding='utf-8') as file:
                return file.read()
        except FileNotFoundError:
            return f"錯誤: 檔案不存在 '{txt_path}'。"
        except UnicodeDecodeError:
            return f"錯誤: 檔案 '{txt_path}' 不是 UTF-8 編碼，請轉換後再使用。"
        except Exception as e:
            return f"讀取 TXT 檔案時發生錯誤: {e}"

    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """
        從 PDF 檔案中提取文字內容。
        這是內部使用的輔助方法。
        """
        text = ""
        try:
            with open(pdf_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                for page_num in range(len(reader.pages)):
                    page = reader.pages[page_num]
                    # 嘗試提取文字，如果頁面沒有文字內容則跳過
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n" # 每頁之間加一個換行符
        except PyPDF2.errors.PdfReadError:
            return f"錯誤: 無法讀取 PDF 檔案 '{pdf_path}'，可能檔案已損壞或加密。"
        except FileNotFoundError:
            return f"錯誤: 檔案不存在 '{pdfㄇ_path}'。"
        except Exception as e:
            return f"提取 PDF 文字時發生未知錯誤: {e}"
        return text

    def attach_pdf(self, pdf_path: str) -> str:
        """
        附加一個 PDF 檔案，提取其內容並儲存以供後續對話使用。
        """
        if not os.path.exists(pdf_path):
            return f"錯誤: 指定的 PDF 檔案不存在: '{pdf_path}'"
        if not pdf_path.lower().endswith('.pdf'):
            return f"錯誤: 檔案 '{pdf_path}' 不是一個 PDF 檔案。"

        print(f"📄 正在從 '{pdf_path}' 提取文字內容，請稍候...")
        extracted_text = self._extract_text_from_pdf(pdf_path)

        if extracted_text.startswith("錯誤:"):
            self.attached_pdf_content = ""
            self.pdf_attached_flag = False
            return extracted_text # 返回錯誤訊息
        elif not extracted_text.strip():
            self.attached_pdf_content = ""
            self.pdf_attached_flag = False
            return "⚠️ 警告: 從 PDF 中提取的文字內容為空。請檢查 PDF 檔案是否包含可讀文字。"
        else:
            self.attached_pdf_content = extracted_text
            self.pdf_attached_flag = True
            # 可選：將 PDF 內容作為系統訊息添加到對話歷史的開頭，這樣模型在每次對話時都會看到它
            # 但更好的做法是在每次發送訊息時，將其作為一個特殊的上下文訊息傳遞
            # self.conversation_history.insert(0, {"role": "system", "content": f"以下是參考文件內容：\n{self.attached_pdf_content}"})
            return f"✅ PDF 檔案 '{pdf_path}' 的內容已成功附加為對話上下文。"
        
    def attach_txt(self, txt_path: str) -> str:
        """
        附加一個 TXT 純文字檔案作為對話上下文。
        """
        if not os.path.exists(txt_path):
            return f"錯誤: 指定的 TXT 檔案不存在: '{txt_path}'"
        if not txt_path.lower().endswith('.txt'):
            return f"錯誤: 檔案 '{txt_path}' 不是一個 TXT 檔案。"
    
        print(f"📄 正在從 '{txt_path}' 讀取文字內容...")
        extracted_text = self._extract_text_from_txt(txt_path)
    
        if extracted_text.startswith("錯誤:") or not extracted_text.strip():
            self.attached_pdf_content = ""
            self.pdf_attached_flag = False
            return extracted_text or "⚠️ 警告: TXT 檔案為空。"
        else:
            self.attached_pdf_content = extracted_text
            self.pdf_attached_flag = True
            return f"✅ TXT 檔案 '{txt_path}' 的內容已成功附加為對話上下文。"

    def send_message(self, message: str) -> str:
        """
        發送訊息給LLM並獲取回應。
        如果已附加 PDF，則將 PDF 內容作為上下文傳遞。
        
        Args:
            message: 用戶輸入的訊息
            
        Returns:
            LLM的回應
        """
        # 準備要發送給 Ollama 的訊息列表
        messages_to_send = []

        # 如果有附加 PDF 內容，將其作為系統提示添加到訊息列表的開頭
        if self.pdf_attached_flag and self.attached_pdf_content:
            # 這裡將 PDF 內容作為一個獨立的系統訊息傳遞，而不是直接修改用戶訊息
            # 這樣可以更好地區分用戶輸入和文件上下文
            messages_to_send.append({
                "role": "system",
                "content": f"以下是參考文件內容：\n\n{self.attached_pdf_content}\n\n請根據這些內容來回答問題，如果問題與文件無關，請說明。"
            })
            
        # 將當前的對話歷史添加到訊息列表
        messages_to_send.extend(self.conversation_history)
        
        # 將用戶當前訊息添加到訊息列表
        messages_to_send.append({"role": "user", "content": message})
        
        # 準備請求資料
        payload = {
            "model": self.model_name,
            "messages": messages_to_send, # 使用包含 PDF 上下文和歷史的訊息列表
            "stream": False
        }
        # 顯示將送出的訊息內容（前200個字）
        
#        debug_preview = "\n".join([f"{m['role']}: {m['content']}" for m in messages_to_send])
#        print(f"\n🧾 傳送前的上下文（預覽前2000字）:\n{debug_preview[:2000]}")
        
        try:
            # 發送請求到Ollama
            response = requests.post(
                self.chat_url,
                json=payload,
                headers={"Content-Type": "application/json"},
                timeout=30000 # 增加超時時間，因為處理 PDF 可能需要更長時間
            )
            
            if response.status_code == 200:
                result = response.json()
                assistant_message = result["message"]["content"]
                
                # 將用戶訊息和助手回應添加到對話歷史（僅用於顯示和後續對話記憶）
                self.conversation_history.append({"role": "user", "content": message})
                self.conversation_history.append({
                    "role": "assistant", 
                    "content": assistant_message
                })
                
                return assistant_message
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
                
        except requests.exceptions.RequestException as e:
            return f"連接錯誤: {str(e)}"
        except json.JSONDecodeError as e:
            return f"JSON解析錯誤: {str(e)}"
        except Exception as e:
            return f"未知錯誤: {str(e)}"
    
    def new_session(self):
        """
        重置對話記憶，開始新的對話會話，並清除附加的 PDF 內容。
        """
        self.conversation_history = []
        self.attached_pdf_content = "" # 清除附加的 PDF 內容
        self.pdf_attached_flag = False # 重置標記
        print("✅ 對話記憶已重置，開始新的會話！")
    
    def show_history(self):
        """
        顯示當前對話歷史
        """
        if not self.conversation_history:
            print("📝 目前沒有對話記錄")
            return
            
        print("\n📚 對話歷史:")
        print("-" * 50)
        for i, msg in enumerate(self.conversation_history, 1):
            role = "用戶" if msg["role"] == "user" else "助手"
            print(f"{i}. {role}: {msg['content']}")
        print("-" * 50)
    
    def check_model(self) -> bool:
        """
        檢查模型是否可用
        
        Returns:
            模型是否可用
        """
        try:
            models_url = f"{self.base_url}/api/tags"
            response = requests.get(models_url, timeout=10)
            
            if response.status_code == 200:
                models = response.json().get("models", [])
                available_models = [model["name"] for model in models]
                
                if self.model_name in available_models:
                    print(f"✅ 模型 {self.model_name} 已就緒")
                    return True
                else:
                    print(f"❌ 模型 {self.model_name} 不可用")
                    print(f"可用模型: {available_models}")
                    return False
            else:
                print(f"❌ 無法連接到Ollama服務: HTTP {response.status_code}")
                return False
                
        except Exception as e:
            print(f"❌ 檢查模型時發生錯誤: {str(e)}")
            return False

def main(model_name="llama3:8b"):
    """
    主要執行函數
    """
    print(f"🤖 Ollama {model_name} 聊天機器人")
    print("=" * 50)
    
    # 初始化聊天機器人
    chatbot = OllamaChatbot(model_name)
    
    # 檢查模型可用性
    if not chatbot.check_model():
        print("\n請確保:")
        print("1. Ollama服務正在運行 (ollama serve)")
        print("2. 已下載llama3模型 (ollama pull llama3)")
        return
    
    print("\n使用說明:")
    print("• 直接輸入訊息與AI對話")
    print("• 輸入 'New Session' 重置對話記憶並清除附加的 PDF")
    print("• 輸入 'history' 查看對話歷史")
    print("• 輸入 'attach_pdf <檔案路徑>' 附加 PDF 檔案作為上下文 (例如: attach_pdf my_document.pdf)")
    print("• 輸入 'attach_txt <檔案路徑>' 附加 TXT 純文字檔案作為上下文 (例如: attach_txt my_notes.txt)")
    print("• 輸入 'quit' 或 'exit' 退出程式")
    print("-" * 50)
    
    # 主對話迴圈
    while True:
        try:
            user_input = input("\n👤 您: ").strip()
            
            # 退出指令
            if user_input.lower() in ['quit', 'exit', '退出', 'q']:
                print("👋 再見！")
                break
            
            # 空輸入處理
            if not user_input:
                print("⚠️  請輸入訊息")
                continue
            
            # 新會話指令
            if user_input.lower() == 'new session':
                chatbot.new_session()
                continue
            
            # 查看歷史指令
            if user_input.lower() == 'history':
                chatbot.show_history()
                continue
            
            # 附加 PDF 指令
            if user_input.lower().startswith('attach_pdf '):
                pdf_path = user_input[len('attach_pdf '):].strip()
                result_message = chatbot.attach_pdf(pdf_path)
                print(result_message)
                continue # 處理完 PDF 後繼續等待用戶輸入
                
            # 附加 TXT 指令
            if user_input.lower().startswith('attach_txt '):
                txt_path = user_input[len('attach_txt '):].strip()
                result_message = chatbot.attach_txt(txt_path)
                print(result_message)
                continue
            
            # 發送訊息並獲取回應
            print("🤔 思考中...")
            response = chatbot.send_message(user_input)
            print(f"🤖 助手: {response}")
            
        except KeyboardInterrupt:
            print("\n\n👋 程式已中斷，再見！")
            break
        except Exception as e:
            print(f"❌ 發生錯誤: {str(e)}")

if __name__ == "__main__":
    main(model_name="gpt-oss:20b")

🤖 Ollama gpt-oss:20b 聊天機器人
✅ 模型 gpt-oss:20b 已就緒

使用說明:
• 直接輸入訊息與AI對話
• 輸入 'New Session' 重置對話記憶並清除附加的 PDF
• 輸入 'history' 查看對話歷史
• 輸入 'attach_pdf <檔案路徑>' 附加 PDF 檔案作為上下文 (例如: attach_pdf my_document.pdf)
• 輸入 'attach_txt <檔案路徑>' 附加 TXT 純文字檔案作為上下文 (例如: attach_txt my_notes.txt)
• 輸入 'quit' 或 'exit' 退出程式
--------------------------------------------------



👤 您:  attach_txt 114_碩士班招生簡章_分頁.txt


📄 正在從 '114_碩士班招生簡章_分頁.txt' 讀取文字內容...
✅ TXT 檔案 '114_碩士班招生簡章_分頁.txt' 的內容已成功附加為對話上下文。



👤 您:  世新大學114學年度碩士班網路報名日期是?


🤔 思考中...
🤖 助手: 


...

✅ 對話記憶已重置，開始新的會話！



👤 您:  === 第 19 頁 === 14    系所代碼  660  系所組別  資訊管理學系碩士班（一般生）  招生名額  9  報考資格  一、大學畢業具有學士學位者（含應屆）。  二、具同等學力資格者：（認定標準詳見本簡章附錄一）  (一)大學肄業持有修業證明書或休學證明書，並檢附歷年成績單。  (二)專科學校畢業者。  (三)高等考試或相當於高等考試之特種考試及格，持有及格證書者。  系所指定 備審資料  書面審查一：  （一）自傳（含求學動機與目標）  （二）讀書計畫  書面審查二：    （一）最高學歷歷年成績單（含學業總排名）正本一份    （二）其他有助審查之資料（如專題作品、專業證照、社團經歷、語言測驗成績、 獎學金證明等）  以上資料請於 114 年 1 月 17 日報名截止日前，併同報名表件寄（送）至招生委員會。  考試項目  書面審查  成績計算 方式  書面審查一×【50 %】+書面審查二×【50 %】= 總成績  錄取同分  參酌順序  一、書面審查一  二、書面審查二  其他規定  一、 本系另辦理碩士班甄試入學，如招生名額不足，得流用至本次考試入學招生。  二、 招生名額含半導體、AI、機械領域擴充名額 1 名。  系所聯絡 方式  系所電話：(02)22368225-63342 系秘書  電子信箱：im@mail.shu.edu.tw  系所網址：https://im.wp.shu.edu.tw/


🤔 思考中...
🤖 助手: Below is a quick‑look English summary of the information you pasted (Page 19 of the admissions booklet).  If there’s anything more specific you’d like—e.g., how to write the “自傳” or what documents you need for the “書面審查二”—just let me know!

| Item | Details |
|------|---------|
| **Program** | Master’s Program in Information Management (General Student) |
| **Quota** | 9 seats (including a special seat reserved for semiconductor, AI, or mechanical‑engineering extensions) |
| **Eligibility** | 1. Bachelor’s degree holders (including those graduating that same year).  <br>2. Persons with “equivalent academic standing”:<br>&nbsp;&nbsp;(a) University students who withdrew/failed but have a certificate of attendance and a transcript for all years.<br>&nbsp;&nbsp;(b) Junior college graduates.<br>&nbsp;&nbsp;(c) Those who passed the Higher‑Level Examination or an equivalent special exam. |
| **Required Documents (by 1 Jan 2025)** | • Application form <br>• “自傳” (personal statem


👤 您:  資管系招生名額多少?


🤔 思考中...
🤖 助手: **資管系（資訊管理學系碩士班）招生名額：**  
- **總名額為 9 個席位。**  
- 在「其他規定」中提到，這其中已經包含了半導體、AI 或機械領域的擴充名額 **1 名**（即 8 個常規席位 + 1 個特殊領域席位 = 9）。  

如果還有任何關於申請流程或其他細節想了解，隨時告訴我！
